# Projection-Based Modes of Variability Processing

This notebook runs a restartable ESP-Lab/PCMDI-style modes-of-variability workflow for observations, E3SM, and CESM-SMYLE, with an S2D-safe design:

1. **Define EOF spatial patterns from a long observation/reanalysis base period.**
2. **Project 2-year E3SM/SMYLE hindcasts onto that fixed observed EOF basis.**
3. **Write projected indices, model-associated regression patterns, and regression significance diagnostics for downstream skill analysis.**

The central reason is that 2-year hindcasts are too short to robustly estimate EOF patterns independently, but they are still useful for evaluating projection-based mode indices and their associated fields.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


## Configuration

Change processing choices only in this cell. Pressure modes use seasonal PSL and temperature modes use monthly SST.

**S2D design choice:** two-year hindcasts should not be used to refit EOF patterns. The notebook uses observations/reanalysis over a selectable base period to establish EOF patterns, then projects E3SM/SMYLE anomalies onto the fixed basis. Model patterns are diagnosed afterward using regression of model fields onto the projected model index.

By default, `eof_reference_start_year=None` and `eof_reference_end_year=None`, so the notebook automatically uses the model period (`start_year`–`end_year`) as the observed EOF training period. Set explicit reference years only for a long-observation sensitivity test.


In [2]:
# Official PMP extra-tropical modes plus ESP-Lab extensions.
OFFICIAL_PMP_MODES = ["NAM", "NAO", "SAM", "PNA", "NPO", "PDO", "NPGO"]
EXTENSION_MODES = ["EA", "SCA", "AMO"]
ALL_MODES = OFFICIAL_PMP_MODES + EXTENSION_MODES

CONFIG = {
    "outdir": Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability"),
    "modes": ALL_MODES,
    "sources": ["obs", "e3sm", "smyle"],
    "init_months": [5, 11],
    "start_year": 1980,
    "end_year": 2018,
    "clim_start": 1980,
    "clim_end": 2010,
    "obs_start_year": 1979,
    "obs_end_year": 2019,
    "monthly_nlead": 24,
    "target_dlat": 2.5,
    "target_dlon": 2.5,

    # EOF/projection strategy.
    # Rationale: each S2D hindcast has only 24 monthly leads, which is too short
    # to estimate a stable EOF pattern. Use observations/reanalysis to establish
    # the EOF basis, then project E3SM/SMYLE hindcast anomalies onto that basis.
    "eof_strategy": "fixed_obs_projection",
    "eof_reference_source": "obs",
    # Leave as None to dynamically inherit start_year/end_year.
    # This keeps the observed EOF training period matched to the model period by default.
    "eof_reference_start_year": 1980,
    "eof_reference_end_year": 2018,
    "require_fixed_obs_projection": True,
    "min_years_for_independent_hindcast_eof": 10,

    # Optional notebook-side regression/metric products.
    # The workflow already writes projected indices and regression-pattern
    # diagnostics. Keep this disabled for pure diagnostic-data production;
    # use the downstream analysis module for skill metrics and plots.
    "compute_regression_patterns": False,
    "regression_index_variable_candidates": [
        "mode_index_projected", "mode_index", "pc", "PC", "index"
    ],
    "regression_field_variable_candidates": [
        "SST_anom", "PSL_anom", "sst_anom", "psl_anom", "anomaly", "anomaly_field", "sst", "SST", "psl", "PSL", "field"
    ],
    "regression_alpha": 0.05,
    "regression_confidence": 0.95,
    "regression_fdr": True,
    "regression_standardize_index": True,
    "regression_output_subdir": "regression_patterns",
    "metrics_output_name": "projection_regression_metrics.csv",

    # Bootstrap of EOF patterns should be disabled for 2-year hindcast samples.
    # Uncertainty should be estimated on projected indices/skill and regression
    # pattern metrics, not by refitting hindcast EOFs.
    "eof_bootstrap_iterations": 0,
    "eof_bootstrap_seed": 42,
    "eof_bootstrap_confidence": 0.90,

    # Dask/workflow controls.
    # dask_workers is the total worker budget you want to use from this notebook.
    # parallel_jobs controls how many independent workflow subprocesses run at once.
    # workers_per_command is computed below unless explicitly set.
    "dask_workers": 8,
    "parallel_jobs": 2,
    "workers_per_command": None,
    "processing_retries": 1,
    "use_dask_scheduler": True,
    "e3sm_nens": 10,
    "smyle_nens": 20,
    "force": False,
}

# Use True only after reviewing the dry run.
RUN_PROCESSING = True


## Build Command

In [3]:
def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "workflows" / "run_modes_of_variability.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from within the ESP-Lab repository.")


def normalize_config(config):
    """Return a copy with derived defaults filled in.

    By default, the observed EOF reference period is matched to the model
    verification period. Override eof_reference_start_year/end_year only for a
    long-record sensitivity test.
    """
    normalized = dict(config)
    if normalized.get("eof_reference_start_year") is None:
        normalized["eof_reference_start_year"] = normalized["start_year"]
    if normalized.get("eof_reference_end_year") is None:
        normalized["eof_reference_end_year"] = normalized["end_year"]
    return normalized


def validate_config(config):
    supported_modes = {
        "NAM", "NAO", "SAM", "PSA1", "PSA2", "PNA", "NPO",
        "EA", "SCA", "PDO", "NPGO", "AMO",
    }
    supported_sources = {"obs", "e3sm", "smyle"}
    unknown_modes = set(config["modes"]) - supported_modes
    unknown_sources = set(config["sources"]) - supported_sources
    if unknown_modes:
        raise ValueError(f"Unsupported modes: {sorted(unknown_modes)}")
    if unknown_sources:
        raise ValueError(f"Unsupported sources: {sorted(unknown_sources)}")
    if not all(1 <= month <= 12 for month in config["init_months"]):
        raise ValueError("Initialization months must be between 1 and 12.")
    if config["start_year"] > config["end_year"]:
        raise ValueError("start_year must not exceed end_year.")
    if config["obs_start_year"] > config["obs_end_year"]:
        raise ValueError("obs_start_year must not exceed obs_end_year.")
    if config["clim_start"] > config["clim_end"]:
        raise ValueError("clim_start must not exceed clim_end.")
    if not (
        config["start_year"] <= config["clim_start"]
        <= config["clim_end"] <= config["end_year"]
    ):
        raise ValueError("Climatology must lie within model processing years.")
    if not (
        config["obs_start_year"] <= config["clim_start"]
        <= config["clim_end"] <= config["obs_end_year"]
    ):
        raise ValueError("Climatology must lie within observation years.")
    if config["eof_reference_source"] != "obs":
        raise ValueError("For this S2D workflow, eof_reference_source should be 'obs'.")
    if not (
        config["obs_start_year"] <= config["eof_reference_start_year"]
        <= config["eof_reference_end_year"] <= config["obs_end_year"]
    ):
        raise ValueError("EOF reference period must lie within observation years.")
    reference_years = config["eof_reference_end_year"] - config["eof_reference_start_year"] + 1
    if reference_years < config["min_years_for_independent_hindcast_eof"]:
        raise ValueError("EOF reference period is too short for a stable observed EOF basis.")
    hindcast_lead_years = config["monthly_nlead"] / 12
    if config["eof_strategy"] == "fixed_obs_projection" and hindcast_lead_years < config["min_years_for_independent_hindcast_eof"]:
        print(
            "Using fixed observed EOF projection: "
            f"monthly_nlead={config['monthly_nlead']} gives only {hindcast_lead_years:.1f} years per hindcast."
        )
    for name in (
        "monthly_nlead", "dask_workers", "e3sm_nens", "smyle_nens", "parallel_jobs"
    ):
        if config[name] < 1:
            raise ValueError(f"{name} must be positive.")
    if config.get("workers_per_command") is not None and config["workers_per_command"] < 1:
        raise ValueError("workers_per_command must be None or positive.")
    if config.get("processing_retries", 0) < 0:
        raise ValueError("processing_retries must be nonnegative.")
    if config["target_dlat"] <= 0 or config["target_dlon"] <= 0:
        raise ValueError("Target-grid spacing must be positive.")
    if config["eof_bootstrap_iterations"] < 0:
        raise ValueError("eof_bootstrap_iterations must be nonnegative.")
    if not 0 < config["eof_bootstrap_confidence"] < 1:
        raise ValueError("eof_bootstrap_confidence must lie between 0 and 1.")
    if not isinstance(config["outdir"], Path):
        raise TypeError("outdir must be a pathlib.Path.")


def build_command(config, dry_run=False):
    command = [
        sys.executable,
        str(REPO_ROOT / "workflows" / "run_modes_of_variability.py"),
        "--outdir", str(config["outdir"]),
        "--modes", *config["modes"],
        "--sources", *config["sources"],
        "--init-months", *map(str, config["init_months"]),
        "--start-year", str(config["start_year"]),
        "--end-year", str(config["end_year"]),
        "--clim-start", str(config["clim_start"]),
        "--clim-end", str(config["clim_end"]),
        "--obs-start-year", str(config["obs_start_year"]),
        "--obs-end-year", str(config["obs_end_year"]),
        "--monthly-nlead", str(config["monthly_nlead"]),
        "--target-dlat", str(config["target_dlat"]),
        "--target-dlon", str(config["target_dlon"]),
        "--dask-workers", str(config["dask_workers"]),
        "--e3sm-nens", str(config["e3sm_nens"]),
        "--smyle-nens", str(config["smyle_nens"]),
        "--eof-bootstrap-iterations", str(config["eof_bootstrap_iterations"]),
        "--eof-bootstrap-seed", str(config["eof_bootstrap_seed"]),
        "--eof-bootstrap-confidence", str(config["eof_bootstrap_confidence"]),
        "--regression-confidence", str(config["regression_confidence"]),
    ]

    command.extend(["--eof-strategy", str(config["eof_strategy"])])
    if config.get("eof_strategy") == "fixed_obs_projection":
        command.extend(
            [
                "--eof-reference-source", str(config["eof_reference_source"]),
                "--eof-reference-start-year", str(config["eof_reference_start_year"]),
                "--eof-reference-end-year", str(config["eof_reference_end_year"]),
            ]
        )

    if config.get("force"):
        command.append("--force")
    if config.get("merge_manifest"):
        command.append("--merge-manifest")
    if dry_run:
        command.append("--dry-run")
    return command


def config_for_task(config, *, modes, dask_workers=None):
    """Return a shallow task-specific config without mutating CONFIG."""
    task_config = dict(config)
    task_config["modes"] = list(modes)
    task_config["merge_manifest"] = True
    if dask_workers is not None:
        task_config["dask_workers"] = int(dask_workers)
    return task_config


def build_processing_tasks(config):
    """Split workflow into restartable, independent tasks.

    Modes sharing the same source field are kept in one command so the runner can
    reuse prepared fields and observational references without concurrent writes.
    """
    total_workers = int(config["dask_workers"])
    pressure_modes = {"NAM", "NAO", "SAM", "PSA1", "PSA2", "PNA", "NPO", "EA", "SCA"}
    groups = [
        ("pressure_modes", [mode for mode in config["modes"] if mode in pressure_modes]),
        ("temperature_modes", [mode for mode in config["modes"] if mode not in pressure_modes]),
    ]
    groups = [(label, modes) for label, modes in groups if modes]
    parallel_jobs = min(int(config["parallel_jobs"]), total_workers, len(groups))
    workers_per_command = config.get("workers_per_command")
    if workers_per_command is None:
        workers_per_command = max(1, total_workers // parallel_jobs)
    workers_per_command = int(workers_per_command)

    tasks = []
    for label, modes in groups:
        task_config = config_for_task(
            config,
            modes=modes,
            dask_workers=workers_per_command,
        )
        tasks.append({"label": label, "command": build_command(task_config)})

    return tasks, parallel_jobs, workers_per_command


REPO_ROOT = find_repo_root()
CONFIG = normalize_config(CONFIG)
validate_config(CONFIG)
COMMAND = build_command(CONFIG)
PROCESSING_TASKS, PARALLEL_JOBS, WORKERS_PER_COMMAND = build_processing_tasks(CONFIG)

print("Repository:", REPO_ROOT)
print("Python:", sys.executable)
print("Output:", CONFIG["outdir"])
print(
    "EOF strategy:",
    f"{CONFIG['eof_strategy']} from {CONFIG['eof_reference_source']} "
    f"{CONFIG['eof_reference_start_year']}-{CONFIG['eof_reference_end_year']}"
)
print("Full command:\n", " ".join(map(str, COMMAND)))
print(f"Split tasks: {len(PROCESSING_TASKS)}")
print(f"Parallel jobs: {PARALLEL_JOBS}")
print(f"Dask workers per command: {WORKERS_PER_COMMAND}")
if CONFIG.get("force") and RUN_PROCESSING:
    print("WARNING: force=True will rebuild compatible cached products.")


Using fixed observed EOF projection: monthly_nlead=24 gives only 2.0 years per hindcast.
Repository: /global/u2/z/zhan391/code/ESP-Lab
Python: /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
Output: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability
EOF strategy: fixed_obs_projection from obs 1980-2018
Full command:
 /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python /global/u2/z/zhan391/code/ESP-Lab/workflows/run_modes_of_variability.py --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability --modes NAM NAO SAM PNA NPO PDO NPGO EA SCA AMO --sources obs e3sm smyle --init-months 5 11 --start-year 1980 --end-year 2018 --clim-start 1980 --clim-end 2010 --obs-start-year 1979 --obs-end-year 2019 --monthly-nlead 24 --target-dlat 2.5 --target-dlon 2.5 --dask-workers 8 --e3sm-nens 10 --smyle-nens 20 --eof-bootstrap-iterations 0 --eof-bootstrap-seed 42 --eof-bootstrap-confidence 0.9 --regression-confidence 0.95 --eof-strategy fixed_obs_projection --eof-refer

## Dry Run

This validates the full configuration first, then prints the planned split tasks. The actual dry-run command still uses the workflow's own `--dry-run` mode and does not write products.

The updated `workflows/run_modes_of_variability.py` supports `--eof-strategy fixed_obs_projection` and the EOF reference-period options. The notebook treats those flags as required by default so accidental fallback to hindcast EOF fitting is avoided.


In [4]:
dry_run = subprocess.run(
    build_command(CONFIG, dry_run=True),
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(dry_run.stdout)
if dry_run.stderr:
    print(dry_run.stderr)
dry_run.check_returncode()

print("\nPlanned split tasks:")
for task in PROCESSING_TASKS:
    print(f"- {task['label']}: {' '.join(map(str, task['command']))}")



INFO: Modes: NAM, NAO, SAM, PNA, NPO, PDO, NPGO, EA, SCA, AMO
INFO: Sources: obs, e3sm, smyle
INFO: Output: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability
INFO: EOF strategy: fixed_obs_projection
INFO: EOF reference period: 1980-2018
INFO: [dry-run] NAM ERA5 field: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/fields/era5_psl_seasonal_2p5x2p5deg.nc
INFO: [dry-run] NAM ERA5 index: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/modes/nam/indices/era5_nam_reference.nc
INFO: [dry-run] NAM E3SM init 05 field: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/fields/e3sm_init05_psl_seasonal_2p5x2p5deg.nc
INFO: [dry-run] NAM E3SM init 05 index: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/modes/nam/indices/e3sm_init05_nam.nc
INFO: [dry-run] NAM E3SM init 11 field: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/fields/e3sm_init11_psl_seasonal_2p5x2p5deg.nc
INFO: [dry-run] NAM E3SM init 11 index: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_

## Run Processing

This cell runs independent workflow tasks through a Dask scheduler. Each task has its own command, live output prefix, and log file. Pressure modes and temperature modes run separately so each shared source field is prepared only once. Existing products are reused unless `force` is enabled.

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed as futures_as_completed


def make_subprocess_environment():
    environment = os.environ.copy()
    environment.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")
    environment.setdefault("PYTHONUNBUFFERED", "1")

    # Prevent nested thread oversubscription when multiple subprocesses run.
    environment["OMP_NUM_THREADS"] = "1"
    environment["OPENBLAS_NUM_THREADS"] = "1"
    environment["MKL_NUM_THREADS"] = "1"
    environment["NUMEXPR_NUM_THREADS"] = "1"
    environment["VECLIB_MAXIMUM_THREADS"] = "1"
    return environment


def terminate_process_group(process, timeout=15):
    if process.poll() is not None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
        process.wait(timeout=timeout)
    except ProcessLookupError:
        return
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()


def run_with_live_output(command, label=None, log_dir=None, retries=0):
    prefix = f"[{label}] " if label else ""
    environment = make_subprocess_environment()

    log_path = None
    if log_dir is not None:
        log_dir = Path(log_dir)
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"{label or 'workflow'}.log"

    last_error = None

    for attempt in range(retries + 1):
        if attempt:
            print(f"{prefix}Retry {attempt}/{retries}", flush=True)
            time.sleep(min(30, 5 * attempt))

        start_time = time.time()
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            env=environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )

        try:
            if process.stdout is None:
                raise RuntimeError("Failed to capture workflow output.")

            log_handle = open(log_path, "a") if log_path else None
            try:
                if log_handle:
                    log_handle.write(f"\n===== START {label} attempt={attempt} =====\n")
                    log_handle.write("COMMAND: " + " ".join(map(str, command)) + "\n")
                    log_handle.flush()

                for line in process.stdout:
                    message = f"{prefix}{line}"
                    print(message, end="", flush=True)
                    if log_handle:
                        log_handle.write(message)
                        log_handle.flush()
            finally:
                if log_handle:
                    log_handle.close()

            return_code = process.wait()

        except BaseException:
            terminate_process_group(process)
            raise

        elapsed_min = (time.time() - start_time) / 60
        if return_code == 0:
            print(f"{prefix}Finished in {elapsed_min:.2f} min", flush=True)
            return {
                "label": label,
                "return_code": return_code,
                "elapsed_min": elapsed_min,
                "log": str(log_path) if log_path else None,
            }

        last_error = subprocess.CalledProcessError(return_code, command)
        print(
            f"{prefix}Failed with return code {return_code} "
            f"after {elapsed_min:.2f} min",
            flush=True,
        )

    raise last_error


def run_tasks_with_dask(tasks, parallel_jobs, log_dir, retries=0):
    from dask.distributed import Client, LocalCluster, as_completed

    cluster = LocalCluster(
        n_workers=parallel_jobs,
        threads_per_worker=1,
        processes=True,
        dashboard_address=":0",
    )
    client = Client(cluster)
    print("Dask dashboard:", client.dashboard_link)

    try:
        futures = [
            client.submit(
                run_with_live_output,
                task["command"],
                label=task["label"],
                log_dir=log_dir,
                retries=retries,
                pure=False,
            )
            for task in tasks
        ]

        results = []
        for future in as_completed(futures):
            result = future.result()
            results.append(result)
            print(f"[DONE] {result['label']} log={result['log']}", flush=True)
        return results

    finally:
        client.close()
        cluster.close()


def run_tasks_with_threads(tasks, parallel_jobs, log_dir, retries=0):
    with ThreadPoolExecutor(max_workers=parallel_jobs) as executor:
        futures = [
            executor.submit(
                run_with_live_output,
                task["command"],
                label=task["label"],
                log_dir=log_dir,
                retries=retries,
            )
            for task in tasks
        ]

        results = []
        for future in futures_as_completed(futures):
            result = future.result()
            results.append(result)
            print(f"[DONE] {result['label']} log={result['log']}", flush=True)
        return results


if RUN_PROCESSING:
    LOG_DIR = CONFIG["outdir"] / "workflow_logs"
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Launching {len(PROCESSING_TASKS)} split tasks")
    print(f"Parallel jobs: {PARALLEL_JOBS}")
    print(f"Dask workers per workflow command: {WORKERS_PER_COMMAND}")
    print(f"Logs: {LOG_DIR}")

    if CONFIG.get("use_dask_scheduler", True):
        try:
            PROCESSING_RESULTS = run_tasks_with_dask(
                PROCESSING_TASKS,
                parallel_jobs=PARALLEL_JOBS,
                log_dir=LOG_DIR,
                retries=CONFIG.get("processing_retries", 0),
            )
        except ImportError:
            print("dask.distributed is not available; falling back to ThreadPoolExecutor.")
            PROCESSING_RESULTS = run_tasks_with_threads(
                PROCESSING_TASKS,
                parallel_jobs=PARALLEL_JOBS,
                log_dir=LOG_DIR,
                retries=CONFIG.get("processing_retries", 0),
            )
    else:
        PROCESSING_RESULTS = run_tasks_with_threads(
            PROCESSING_TASKS,
            parallel_jobs=PARALLEL_JOBS,
            log_dir=LOG_DIR,
            retries=CONFIG.get("processing_retries", 0),
        )

    display(pd.DataFrame(PROCESSING_RESULTS).sort_values("label").reset_index(drop=True))

else:
    print(
        "Processing is disabled. "
        "Set RUN_PROCESSING = True to launch the workflow."
    )


Launching 2 split tasks
Parallel jobs: 2
Dask workers per workflow command: 4
Logs: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/workflow_logs
Dask dashboard: http://127.0.0.1:39263/status
[pressure_modes] INFO: Modes: NAM, NAO, SAM, PNA, NPO, EA, SCA
[pressure_modes] INFO: Sources: obs, e3sm, smyle
[pressure_modes] INFO: Output: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability
[pressure_modes] INFO: EOF strategy: fixed_obs_projection
[pressure_modes] INFO: EOF reference period: 1980-2018
[pressure_modes] INFO: NAM ERA5: processing PSL
[pressure_modes] [OBS] Mapping field 'PSL' -> 'psl'
[temperature_modes] INFO: Modes: PDO, NPGO, AMO
[temperature_modes] INFO: Sources: obs, e3sm, smyle
[temperature_modes] INFO: Output: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability
[temperature_modes] INFO: EOF strategy: fixed_obs_projection
[temperature_modes] INFO: EOF reference period: 1980-2018
[temperature_modes] INFO: PDO HadISST2: processing SST
[temperature_modes] [OBS

,label,return_code,elapsed_min,log
0,pressure_modes,0,16.179109,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_va...
1,temperature_modes,0,47.999949,/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_va...


## Product Summary

The expected scientific output is:

- observed EOF spatial pattern and observed PC/index from the selected reference period;
- E3SM/SMYLE projected indices on the observed EOF basis;
- model-associated regression patterns computed from model fields and projected model indices;
- significance masks and pattern metrics for model-vs-observed comparison.

Hindcast products should not be interpreted as independently estimated EOF patterns.

In [6]:
manifest_path = CONFIG["outdir"] / "modes_manifest.json"

if not manifest_path.exists():
    print("No manifest found yet:", manifest_path)
else:
    manifest = json.loads(manifest_path.read_text())
    rows = []
    for product, details in manifest.get("products", {}).items():
        index_path = Path(details["index"])
        field_path = Path(details["field"])
        rows.append({
            "product": product,
            "status": details.get("status", ""),
            "index_exists": index_path.exists(),
            "field_exists": field_path.exists(),
            "index_MB": round(index_path.stat().st_size / 1e6, 2) if index_path.exists() else np.nan,
        })
    display(pd.DataFrame(rows).sort_values("product").reset_index(drop=True))
    print("Manifest:", manifest_path)


,product,status,index_exists,field_exists,index_MB
0,AMO:e3sm_init05,ok,True,True,1.02
1,AMO:e3sm_init11,ok,True,True,1.02
2,AMO:reference,ok,True,True,0.23
3,AMO:smyle_init05,ok,True,True,1.15
4,AMO:smyle_init11,ok,True,True,1.15
5,EA:e3sm_init05,ok,True,True,0.43
6,EA:e3sm_init11,ok,True,True,0.43
7,EA:reference,ok,True,True,0.29
8,EA:smyle_init05,ok,True,True,0.47
9,EA:smyle_init11,ok,True,True,0.47


Manifest: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/modes_manifest.json


## Inspect One Index Product

In [13]:
PRODUCT_TO_INSPECT = "NAO:e3sm_init05"

if not manifest_path.exists():
    print("Run processing before inspecting a product.")
else:
    manifest = json.loads(manifest_path.read_text())
    details = manifest.get("products", {}).get(PRODUCT_TO_INSPECT)
    if details is None:
        print(f"{PRODUCT_TO_INSPECT!r} is not present in the manifest.")
    else:
        product_path = Path(details["index"])
        with xr.open_dataset(product_path) as dataset:
            print(product_path)
            print("Dimensions:", dict(dataset.sizes))
            print("Variables:", list(dataset.data_vars))
            for name in ("mode_index", "mode_index_conventional", "mode_variance_fraction"):
                if name in dataset:
                    array = dataset[name]
                    print(f"{name}: {int(array.notnull().sum())}/{array.size} finite")
            if "target_month" in dataset:
                print("Target months:", dataset["target_month"].values.tolist())


/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/modes/nao/indices/e3sm_init05_nao.nc
Dimensions: {'Y': 39, 'L': 7, 'M': 10, 'lat': 25, 'lon': 53}
Variables: ['mode_index', 'mode_regression_pattern', 'mode_regression_pvalue', 'mode_regression_significant', 'mode_regression_r', 'target_month', 'mode_pattern_rmse_reference', 'mode_pattern_pcc_reference', 'mode_pattern', 'mode_variance_fraction', 'nao_station', 'valid_time', 'nao_eof', 'nao_pattern', 'nao_variance_fraction']
mode_index: 2730/2730 finite
mode_variance_fraction: 0/7 finite
Target months: [7, 10, 1, 4, 7, 10, 1]


## Projection-Based Regression Pattern Diagnostics

This section keeps the scientifically useful model pattern analysis while avoiding unstable hindcast EOF refitting. For each product, the notebook regresses the model or observed anomaly field onto the corresponding projected mode index:

```text
X(x, y, t) = beta(x, y) * PC(t) + residual
```

The resulting `beta` map is the model-associated pattern for the projected mode. The p-value and optional FDR-adjusted significance mask support stippling or masking in metrics plots.

In [14]:
from math import erf, sqrt

REGRESSION_DIR = CONFIG["outdir"] / CONFIG["regression_output_subdir"]
REGRESSION_DIR.mkdir(parents=True, exist_ok=True)


def first_existing_variable(dataset, candidates, required=True):
    for name in candidates:
        if name in dataset.data_vars:
            return name
    if required:
        raise KeyError(
            "None of the candidate variables were found. "
            f"Candidates={candidates}; available={list(dataset.data_vars)}"
        )
    return None


def find_lat_lon_names(dataset):
    lat_candidates = ["lat", "latitude", "LAT", "yc", "Y", "nlat"]
    lon_candidates = ["lon", "longitude", "LON", "xc", "X", "nlon"]
    lat_name = next((name for name in lat_candidates if name in dataset.coords or name in dataset.dims), None)
    lon_name = next((name for name in lon_candidates if name in dataset.coords or name in dataset.dims), None)
    if lat_name is None or lon_name is None:
        raise ValueError(f"Could not infer lat/lon names from dims={dataset.dims} coords={list(dataset.coords)}")
    return lat_name, lon_name


def infer_sample_dims(field, index):
    """Return common non-spatial dimensions used as regression samples."""
    lat_lon = set()
    try:
        lat, lon = find_lat_lon_names(field.to_dataset(name="field"))
        lat_lon.update([lat, lon])
    except Exception:
        pass
    sample_dims = [dim for dim in index.dims if dim in field.dims and dim not in lat_lon]
    if not sample_dims:
        # Fall back to common temporal/ensemble/lead dimensions used by ESP-Lab style products.
        preferred = ["time", "T", "year", "Y", "month", "M", "lead", "L", "member", "ens", "nens"]
        sample_dims = [dim for dim in preferred if dim in index.dims and dim in field.dims]
    if not sample_dims:
        raise ValueError(f"Could not infer common sample dims. field={field.dims}, index={index.dims}")
    return sample_dims


def normal_two_sided_p_from_t(t_value):
    """Normal-approximation two-sided p-value for large-sample regression maps."""
    z = np.abs(t_value)
    return 2.0 * (1.0 - 0.5 * (1.0 + xr.apply_ufunc(lambda x: erf(x / sqrt(2.0)), z)))


def benjamini_hochberg_mask(p_values, alpha=0.05):
    """Benjamini-Hochberg FDR mask implemented without statsmodels."""
    p = np.asarray(p_values.values, dtype=float)
    flat = p.ravel()
    finite = np.isfinite(flat)
    result = np.zeros_like(flat, dtype=bool)
    if finite.sum() == 0:
        return xr.zeros_like(p_values, dtype=bool)

    finite_p = flat[finite]
    order = np.argsort(finite_p)
    ranked = finite_p[order]
    m = ranked.size
    thresholds = alpha * (np.arange(1, m + 1) / m)
    passing = ranked <= thresholds
    if passing.any():
        cutoff = ranked[np.where(passing)[0].max()]
        result[finite] = finite_p <= cutoff
    return xr.DataArray(result.reshape(p.shape), coords=p_values.coords, dims=p_values.dims)


def regression_map(field, index, *, standardize_index=True, alpha=0.05, fdr=True):
    """Regress a gridded field onto a projected mode index.

    Returns beta, correlation, p-value, and significance mask. The regression is
    vectorized over all non-sample dimensions, so it should work for maps with
    lat/lon and optional lead/month/member dimensions after stacking samples.
    """
    field, index = xr.align(field, index, join="inner")
    sample_dims = infer_sample_dims(field, index)
    sample_name = "sample"

    x = field.stack({sample_name: sample_dims}).transpose(sample_name, ...)
    pc = index.stack({sample_name: sample_dims}).transpose(sample_name)

    valid = np.isfinite(pc)
    pc = pc.where(valid)
    x = x.where(valid)

    if standardize_index:
        pc = (pc - pc.mean(sample_name)) / pc.std(sample_name)
    else:
        pc = pc - pc.mean(sample_name)

    x_anom = x - x.mean(sample_name)
    pc_anom = pc - pc.mean(sample_name)

    n = xr.ufuncs.isfinite(x_anom * pc_anom).sum(sample_name)
    cov = (x_anom * pc_anom).sum(sample_name) / xr.where(n > 1, n - 1, np.nan)
    var_pc = (pc_anom ** 2).sum(sample_name) / xr.where(n > 1, n - 1, np.nan)
    var_x = (x_anom ** 2).sum(sample_name) / xr.where(n > 1, n - 1, np.nan)

    beta = cov / var_pc
    r = cov / np.sqrt(var_pc * var_x)
    t = r * np.sqrt((n - 2) / xr.where(1 - r ** 2 > 0, 1 - r ** 2, np.nan))
    p = normal_two_sided_p_from_t(t)
    sig = p < alpha
    if fdr:
        try:
            sig = benjamini_hochberg_mask(p, alpha=alpha)
        except Exception as exc:
            print(f"FDR failed; using pointwise p<{alpha}: {exc}")
            sig = p < alpha

    return xr.Dataset({
        "regression_beta": beta,
        "regression_r": r,
        "regression_p_value": p,
        "regression_significant": sig,
        "regression_sample_size": n,
    })


## Compute Regression Products from the Manifest

This cell scans `modes_manifest.json`, opens each product's projected index and associated anomaly field, then writes one regression-pattern file per product. It is intentionally tolerant of variable-name differences across workflow versions.

In [15]:
def compute_regression_products_from_manifest(manifest_path, config):
    if not Path(manifest_path).exists():
        print("No manifest found yet:", manifest_path)
        return pd.DataFrame()

    manifest = json.loads(Path(manifest_path).read_text())
    rows = []

    for product, details in sorted(manifest.get("products", {}).items()):
        index_path = Path(details.get("index", ""))
        field_path = Path(details.get("field", ""))
        if not index_path.exists() or not field_path.exists():
            rows.append({
                "product": product,
                "status": "missing index or field product",
                "index": str(index_path),
                "field": str(field_path),
            })
            continue

        out_path = REGRESSION_DIR / f"{product.replace(':', '_')}_regression.nc"
        if out_path.exists() and not config.get("force"):
            rows.append({"product": product, "status": "exists", "regression": str(out_path)})
            continue

        try:
            with xr.open_dataset(index_path) as ds_index, xr.open_dataset(field_path) as ds_field:
                index_name = first_existing_variable(
                    ds_index,
                    config["regression_index_variable_candidates"],
                )
                field_name = first_existing_variable(
                    ds_field,
                    config["regression_field_variable_candidates"],
                )
                result = regression_map(
                    ds_field[field_name],
                    ds_index[index_name],
                    standardize_index=config["regression_standardize_index"],
                    alpha=config["regression_alpha"],
                    fdr=config["regression_fdr"],
                )
                result.attrs.update({
                    "product": product,
                    "index_file": str(index_path),
                    "field_file": str(field_path),
                    "index_variable": index_name,
                    "field_variable": field_name,
                    "method": "regression of field anomalies onto fixed-observed-EOF projected index",
                    "eof_strategy": config["eof_strategy"],
                    "eof_reference_source": config["eof_reference_source"],
                    "eof_reference_period": f"{config['eof_reference_start_year']}-{config['eof_reference_end_year']}",
                })
                result.to_netcdf(out_path)

            rows.append({
                "product": product,
                "status": "written",
                "regression": str(out_path),
                "index_variable": index_name,
                "field_variable": field_name,
            })

        except Exception as exc:
            rows.append({
                "product": product,
                "status": f"failed: {type(exc).__name__}: {exc}",
                "index": str(index_path),
                "field": str(field_path),
            })

    return pd.DataFrame(rows)


if CONFIG["compute_regression_patterns"]:
    REGRESSION_PRODUCTS = compute_regression_products_from_manifest(manifest_path, CONFIG)
    display(REGRESSION_PRODUCTS)
else:
    print("Regression-pattern diagnostics disabled by CONFIG['compute_regression_patterns'].")


Regression-pattern diagnostics disabled by CONFIG['compute_regression_patterns'].


## Pattern Metrics for Model Evaluation

For each mode/source/init product, this cell compares model regression patterns against the corresponding observed regression pattern when available. The main metrics are:

- **pattern correlation** between model and observed regression maps;
- **regression amplitude ratio** based on spatial standard deviation;
- **significant-area fraction** for the model regression map.

These are suitable for a compact metrics plot because the model pattern comes from regression onto the projected index, not from an unstable 2-year EOF.

In [16]:
def parse_product_name(product):
    if ":" in product:
        mode, source_tag = product.split(":", 1)
    else:
        mode, source_tag = product, ""
    if "_init" in source_tag:
        source, init = source_tag.split("_init", 1)
        init = f"init{init}"
    else:
        source, init = source_tag, ""
    return mode, source, init


def spatial_dims_from_dataarray(array):
    candidates = ["lat", "latitude", "LAT", "yc", "Y", "nlat", "lon", "longitude", "LON", "xc", "X", "nlon"]
    return [dim for dim in array.dims if dim in candidates]


def pattern_correlation(model_pattern, obs_pattern):
    model_pattern, obs_pattern = xr.align(model_pattern, obs_pattern, join="inner")
    common_dims = [dim for dim in model_pattern.dims if dim in obs_pattern.dims]
    if not common_dims:
        raise ValueError("No common dimensions for pattern correlation.")
    a = model_pattern.stack(space=common_dims)
    b = obs_pattern.stack(space=common_dims)
    valid = np.isfinite(a) & np.isfinite(b)
    a = a.where(valid, drop=True)
    b = b.where(valid, drop=True)
    if a.size < 3:
        return np.nan
    return float(xr.corr(a, b, dim="space"))


def build_pattern_metrics(regression_products):
    if regression_products is None or regression_products.empty:
        return pd.DataFrame()

    good = regression_products[regression_products["status"].isin(["written", "exists"])]
    product_to_path = dict(zip(good["product"], good["regression"]))
    obs_by_mode = {}
    for product, path in product_to_path.items():
        mode, source, init = parse_product_name(product)
        if source == "obs":
            obs_by_mode[mode] = Path(path)

    rows = []
    for product, path in product_to_path.items():
        mode, source, init = parse_product_name(product)
        path = Path(path)
        if source == "obs" or mode not in obs_by_mode:
            continue
        try:
            with xr.open_dataset(path) as ds_model, xr.open_dataset(obs_by_mode[mode]) as ds_obs:
                beta_model = ds_model["regression_beta"]
                beta_obs = ds_obs["regression_beta"]
                pcorr = pattern_correlation(beta_model, beta_obs)
                amp_ratio = float(beta_model.std(skipna=True) / beta_obs.std(skipna=True))
                sig_frac = float(ds_model["regression_significant"].mean(skipna=True))
                n_eff = float(ds_model["regression_sample_size"].median(skipna=True))
                rows.append({
                    "mode": mode,
                    "source": source,
                    "init": init,
                    "pattern_correlation": pcorr,
                    "amplitude_ratio": amp_ratio,
                    "significant_area_fraction": sig_frac,
                    "median_sample_size": n_eff,
                    "regression_file": str(path),
                })
        except Exception as exc:
            rows.append({
                "mode": mode,
                "source": source,
                "init": init,
                "pattern_correlation": np.nan,
                "amplitude_ratio": np.nan,
                "significant_area_fraction": np.nan,
                "median_sample_size": np.nan,
                "regression_file": str(path),
                "status": f"failed: {type(exc).__name__}: {exc}",
            })

    metrics = pd.DataFrame(rows)
    if not metrics.empty:
        metrics_path = CONFIG["outdir"] / CONFIG["metrics_output_name"]
        metrics.to_csv(metrics_path, index=False)
        print("Wrote metrics:", metrics_path)
    return metrics


METRICS = build_pattern_metrics(REGRESSION_PRODUCTS if "REGRESSION_PRODUCTS" in globals() else pd.DataFrame())
display(METRICS)


""


## Metrics Plot

This plot summarizes whether projected-mode regression patterns in E3SM/SMYLE resemble the observed regression patterns. Higher pattern correlation is better; amplitude ratio close to 1 means comparable pattern amplitude.

In [17]:
import matplotlib.pyplot as plt

if "METRICS" not in globals() or METRICS.empty:
    print("No metrics available yet.")
else:
    plot_df = METRICS.copy()
    plot_df["label"] = plot_df["mode"] + " " + plot_df["source"] + " " + plot_df["init"]
    plot_df = plot_df.sort_values(["mode", "source", "init"])

    ax = plot_df.plot.bar(
        x="label",
        y="pattern_correlation",
        legend=False,
        figsize=(max(10, 0.35 * len(plot_df)), 4),
    )
    ax.axhline(0, linewidth=1)
    ax.set_ylabel("Pattern correlation with observed regression map")
    ax.set_xlabel("")
    ax.set_title("Projection-based mode pattern skill")
    plt.xticks(rotation=75, ha="right")
    plt.tight_layout()
    fig_path = CONFIG["outdir"] / "projection_pattern_correlation_metrics.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()


No metrics available yet.


## Inspect One Regression Pattern

Use this cell to quickly check one product and confirm that `regression_beta`, `regression_p_value`, and `regression_significant` exist. The significance mask can be used for stippling in the final map plot.

In [18]:
REGRESSION_PRODUCT_TO_INSPECT = "NAO:e3sm_init05"

reg_path = REGRESSION_DIR / f"{REGRESSION_PRODUCT_TO_INSPECT.replace(':', '_')}_regression.nc"
if not reg_path.exists():
    print("Regression file not found:", reg_path)
else:
    with xr.open_dataset(reg_path) as ds:
        print(reg_path)
        print(ds)
        if "regression_beta" in ds:
            ds["regression_beta"].plot(figsize=(8, 4))
            plt.title(f"{REGRESSION_PRODUCT_TO_INSPECT} regression beta")
            plt.show()
        if "regression_significant" in ds:
            print("Significant fraction:", float(ds["regression_significant"].mean(skipna=True)))


Regression file not found: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/modes_variability/regression_patterns/NAO_e3sm_init05_regression.nc
